# Offline Reinforcement Learning
## Learning from Fixed Datasets — From Behavior Cloning to Conservative Q-Learning

**Topics covered:** Problem setup · Behavior cloning · Offline Q-learning · Extrapolation error · Conservative Q-Learning (CQL) · Importance sampling · Distributional shift analysis

---

## 1. Problem Statement

**Online RL** requires direct environment interaction: the agent collects transitions $(s, a, r, s')$ by acting in the environment and updates its policy accordingly. This works for simulations and games but is costly or dangerous in the real world.

**Offline RL** (also called *batch RL*) learns a policy entirely from a *fixed, pre-collected dataset* $\mathcal{D} = \{(s_i, a_i, r_i, s'_i)\}_{i=1}^N$ without further environment interaction. The dataset is generated by some *behavior policy* $\mu(a|s)$, which may be suboptimal, random, or a mixture of multiple policies.

### Why it matters

| Domain | Why offline RL is needed |
|--------|-------------------------|
| Healthcare | Clinical trial logs, EHR data; cannot experiment on patients |
| Robotics | Robot operation logs; re-running failures is expensive/unsafe |
| Recommendation | Historical click data; live A/B testing is costly |
| Autonomous driving | Driving logs; safety prevents online exploration |

### The core challenge: distributional shift

The behavior policy $\mu$ covers only a subset of $(s, a)$ pairs. When a learned policy $\pi$ queries Q-values for out-of-distribution (OOD) actions — states or actions not seen in $\mathcal{D}$ — function approximation errors accumulate. This *extrapolation error* causes standard offline Q-learning to dramatically overestimate the value of OOD actions, leading to catastrophic policy collapse.

$$
\underbrace{\text{extrapolation error}}_{ \hat{Q}(s,a) \gg Q^*(s,a) \text{ for } (s,a) \notin \mathcal{D}}
\quad \Longrightarrow \quad
\pi(s) = \arg\max_a \hat{Q}(s,a) \text{ picks bad OOD actions}
$$

### Solutions covered in this notebook

1. **Behavior Cloning (BC):** supervised imitation — ignores distributional shift but shows compounding errors
2. **Offline Q-Learning:** demonstrate extrapolation error
3. **Conservative Q-Learning (CQL):** add a regularizer that *penalises* OOD Q-values
4. **Importance Sampling (IS):** correct for the mismatch between $\mu$ and $\pi$ in off-policy evaluation

## 2. Imports and Global Constants

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from collections import defaultdict

# ── Global constants ──────────────────────────────────────────────────────────
SEED       = 42
GRID_SIZE  = 5          # 5×5 grid world
GAMMA      = 0.95       # discount factor
ALPHA      = 0.1        # Q-learning step size
N_ACTIONS  = 4          # up, right, down, left

# Colour palette (consistent with repo style)
C_BLUE   = '#2196F3'
C_GREEN  = '#4CAF50'
C_RED    = '#F44336'
C_ORANGE = '#FF9800'
C_PURPLE = '#9C27B0'
C_GREY   = '#9E9E9E'

rng = np.random.default_rng(SEED)
print("[PASS] Imports and constants loaded")

## 3. Grid World Environment

A **5×5 grid world** identical in structure to the existing DP and TD notebooks:

- States: integer indices $0 \ldots 24$ (row-major)
- Actions: 0=Up, 1=Right, 2=Down, 3=Left
- Goal state: $(4, 4)$ — reward $+10$
- Obstacles: three cells that terminate the episode with reward $-5$
- Step reward: $-0.1$ (encourages shorter paths)
- Episode terminates on reaching goal or obstacle

In [ ]:
class GridWorld:
    """5×5 grid world for offline RL experiments.

    Coordinate convention: (row, col) with (0,0) top-left.
    Actions: 0=Up, 1=Right, 2=Down, 3=Left.
    """

    ACTION_DELTAS = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # up right down left
    ACTION_NAMES  = ['Up', 'Right', 'Down', 'Left']

    def __init__(self, size=GRID_SIZE):
        self.size      = size
        self.n_states  = size * size
        self.n_actions = N_ACTIONS
        self.goal      = (size - 1, size - 1)          # bottom-right
        self.obstacles = {(1, 2), (2, 4), (3, 1)}      # three obstacle cells
        self.start     = (0, 0)
        self._state    = self.start

    # ── Coordinate helpers ──────────────────────────────────────────────────
    def rc_to_s(self, r, c):   return r * self.size + c
    def s_to_rc(self, s):      return divmod(s, self.size)

    @property
    def state(self):           return self.rc_to_s(*self._state)

    # ── Core MDP methods ────────────────────────────────────────────────────
    def reset(self):
        self._state = self.start
        return self.state

    def step(self, action):
        """Take action; return (next_state, reward, done)."""
        r, c = self._state
        dr, dc = self.ACTION_DELTAS[action]
        nr, nc = r + dr, c + dc

        # Clamp to grid boundaries
        nr = max(0, min(self.size - 1, nr))
        nc = max(0, min(self.size - 1, nc))
        self._state = (nr, nc)

        if self._state == self.goal:
            return self.state, 10.0, True
        if self._state in self.obstacles:
            return self.state, -5.0, True
        return self.state, -0.1, False

    def valid_states(self):
        """All non-obstacle, non-goal states (usable as starting states)."""
        all_rc = [(r, c) for r in range(self.size) for c in range(self.size)]
        return [self.rc_to_s(r, c) for r, c in all_rc
                if (r, c) not in self.obstacles and (r, c) != self.goal]


# ── Sanity check ─────────────────────────────────────────────────────────────
env = GridWorld()
s   = env.reset()
s2, r, done = env.step(1)   # move right from (0,0)
assert s == 0 and s2 == 1 and not done, "Step sanity failed"

# Reach goal programmatically: walk right then down
env.reset()
for _ in range(4): env.step(1)   # right ×4 → (0,4)
for _ in range(3): env.step(2)   # down  ×3 → (3,4)
s_goal, r_goal, done_goal = env.step(2)  # down once more → (4,4)
assert done_goal and r_goal == 10.0, "Goal detection failed"

print(f"[PASS] GridWorld: n_states={env.n_states}, "
      f"goal={env.goal}, obstacles={env.obstacles}")

In [ ]:
# ── Figure 1: Grid world layout ───────────────────────────────────────────────
def plot_grid(env, title="Grid World", Q=None, policy=None, ax=None):
    """Visualise the grid world, optionally overlaying Q-values and policy arrows."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5, 5))

    size = env.size
    # Background
    bg = np.zeros((size, size))
    for (r, c) in env.obstacles:
        bg[r, c] = -1
    goal_r, goal_c = env.goal
    bg[goal_r, goal_c] = 1

    cmap = mcolors.ListedColormap([C_RED, '#FAFAFA', C_GREEN])
    ax.imshow(bg, cmap=cmap, vmin=-1, vmax=1, origin='upper')

    # Grid lines
    for x in range(size + 1):
        ax.axhline(x - 0.5, color='k', lw=0.8)
        ax.axvline(x - 0.5, color='k', lw=0.8)

    # Labels
    for r in range(size):
        for c in range(size):
            s = env.rc_to_s(r, c)
            if (r, c) == env.goal:
                ax.text(c, r, 'G', ha='center', va='center',
                        fontsize=14, fontweight='bold', color='white')
            elif (r, c) in env.obstacles:
                ax.text(c, r, 'X', ha='center', va='center',
                        fontsize=14, fontweight='bold', color='white')
            else:
                if Q is not None:
                    v = np.max(Q[s])
                    ax.text(c, r, f'{v:.1f}', ha='center', va='center',
                            fontsize=8, color='#333333')
                else:
                    ax.text(c, r, str(s), ha='center', va='center',
                            fontsize=9, color='#444444')

    # Policy arrows
    if policy is not None:
        arrow_delta = {0: (0, -0.3), 1: (0.3, 0), 2: (0, 0.3), 3: (-0.3, 0)}
        for r in range(size):
            for c in range(size):
                s = env.rc_to_s(r, c)
                if (r, c) in env.obstacles or (r, c) == env.goal:
                    continue
                a = policy[s]
                dc, dr = arrow_delta[a]
                ax.annotate('', xy=(c + dc, r + dr), xytext=(c, r),
                            arrowprops=dict(arrowstyle='->', color=C_BLUE, lw=1.5))

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(range(size))
    ax.set_yticks(range(size))
    ax.set_xticklabels(range(size))
    ax.set_yticklabels(range(size))
    if standalone:
        plt.tight_layout()
        plt.show()


plot_grid(env, title="Grid World Layout  (G=goal, X=obstacle, numbers=state index)")

## 4. Dataset Generation

We generate three offline datasets using different behavior policies:

| Dataset | Behavior policy | Description |
|---------|----------------|-------------|
| `random` | $\mu(a|s) = \text{Uniform}$ | Pure random walk |
| `eps_optimal` | $\epsilon$-greedy w.r.t. optimal Q | Mixed — mostly optimal with exploration |
| `expert` | $\pi^*$ (optimal Q-policy) | Near-optimal demonstrations |

We first compute the **optimal Q-table** via tabular Q-learning with online interaction (this is the *only* place we interact with the environment), then generate the three datasets.

In [ ]:
# ── Online Q-learning to obtain optimal Q* ────────────────────────────────────
def run_online_qlearning(env, n_episodes=5000, alpha=ALPHA, gamma=GAMMA,
                          eps_start=1.0, eps_end=0.05, seed=SEED):
    """Standard online Q-learning with ε-greedy exploration.

    Returns Q-table shape (n_states, n_actions) and per-episode rewards.
    """
    rng_local = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []

    for ep in range(n_episodes):
        eps = eps_start + (eps_end - eps_start) * ep / n_episodes
        s   = env.reset()
        total_r, done = 0.0, False

        while not done:
            # ε-greedy action
            if rng_local.random() < eps:
                a = rng_local.integers(env.n_actions)
            else:
                a = int(np.argmax(Q[s]))

            s2, r, done = env.step(a)
            # Bellman update
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) * (1 - done) - Q[s, a])
            s, total_r = s2, total_r + r

        rewards.append(total_r)

    return Q, rewards


Q_star, online_rewards = run_online_qlearning(env)
pi_star = np.argmax(Q_star, axis=1)

print("[PASS] Online Q-learning complete")
print(f"       Last 500-ep mean reward: {np.mean(online_rewards[-500:]):.3f}")
print(f"       Q*(s=0): {Q_star[0].round(3)}")

In [ ]:
# ── Dataset generation ────────────────────────────────────────────────────────
def collect_dataset(env, policy_fn, n_transitions=10_000, max_steps=50,
                    seed=SEED):
    """Collect a fixed offline dataset using policy_fn(s, rng) -> action.

    Returns list of (s, a, r, s', done) tuples.
    """
    rng_local = np.random.default_rng(seed)
    dataset   = []

    while len(dataset) < n_transitions:
        s    = env.reset()
        done = False
        t    = 0
        while not done and t < max_steps:
            a            = policy_fn(s, rng_local)
            s2, r, done  = env.step(a)
            dataset.append((s, a, r, s2, done))
            s, t = s2, t + 1

    return dataset[:n_transitions]


def random_policy(s, rng):            return int(rng.integers(N_ACTIONS))
def expert_policy(s, rng):            return int(pi_star[s])
def eps_optimal_policy(s, rng, eps=0.3):
    return (int(rng.integers(N_ACTIONS)) if rng.random() < eps
            else int(pi_star[s]))


N_TRANSITIONS = 8_000

datasets = {
    'random':      collect_dataset(env, random_policy,      N_TRANSITIONS, seed=SEED),
    'eps_optimal': collect_dataset(env, eps_optimal_policy, N_TRANSITIONS, seed=SEED+1),
    'expert':      collect_dataset(env, expert_policy,      N_TRANSITIONS, seed=SEED+2),
}

# Dataset statistics
for name, ds in datasets.items():
    rewards = [t[2] for t in ds]
    goals   = sum(1 for t in ds if t[2] == 10.0)
    print(f"  {name:12s}: {len(ds):5d} transitions | "
          f"mean_r={np.mean(rewards):.3f} | goal_hits={goals}")

print("[PASS] Datasets generated")

## 5. Behavior Cloning

**Behavior Cloning (BC)** treats offline RL as supervised classification: given a state $s$, predict the action $a$ taken by the behavior policy.

$$
\mathcal{L}_{\text{BC}}(\theta) = -\mathbb{E}_{(s,a) \sim \mathcal{D}}\left[\log \pi_\theta(a \mid s)\right]
$$

With tabular policies we maintain action count matrices and compute:

$$
\pi_{\text{BC}}(a \mid s) = \frac{\text{count}(s, a) + \epsilon}{\sum_{a'} \text{count}(s, a') + n_{\text{actions}} \cdot \epsilon}
$$

### Failure mode: Compounding errors

BC fails under *covariate shift*: at test time the agent visits states not in $\mathcal{D}$ (because the behavior policy took different actions), where the cloned policy is undefined or wrong. Errors compound over a trajectory of length $T$ — a policy with single-step accuracy $1-\epsilon$ has $T$-step accuracy $(1-\epsilon)^T$.

In [ ]:
# ── Behavior Cloning (tabular) ────────────────────────────────────────────────
def behavior_clone(dataset, n_states, n_actions, smoothing=1e-3):
    """Fit a BC policy by counting (s,a) occurrences in the dataset.

    Returns:
        pi_bc  -- (n_states, n_actions) probability table
        counts -- (n_states, n_actions) raw counts
    """
    counts = np.full((n_states, n_actions), smoothing)
    for (s, a, r, s2, done) in dataset:
        counts[s, a] += 1.0
    pi_bc = counts / counts.sum(axis=1, keepdims=True)
    return pi_bc, counts


def greedy_policy_from_pi(pi):
    """Deterministic greedy policy: argmax of action probabilities."""
    return np.argmax(pi, axis=1)


bc_policies = {}
for name, ds in datasets.items():
    pi_bc, counts = behavior_clone(ds, env.n_states, env.n_actions)
    bc_policies[name] = {'pi': pi_bc, 'counts': counts,
                          'greedy': greedy_policy_from_pi(pi_bc)}

print("[PASS] Behavior cloning complete")

# State coverage: fraction of states with ≥5 visits in the dataset
for name, bcp in bc_policies.items():
    visited = (bcp['counts'].sum(axis=1) > 5).sum()
    print(f"  {name:12s}: {visited}/{env.n_states} states with ≥5 samples")

In [ ]:
# ── Evaluate a deterministic policy by rollout ────────────────────────────────
def evaluate_policy(env, policy_arr, n_episodes=500, max_steps=50, seed=0):
    """Roll out a deterministic policy; return mean episode reward."""
    rng_local = np.random.default_rng(seed)
    ep_rewards = []
    for _ in range(n_episodes):
        s    = env.reset()
        done = False
        tot  = 0.0
        for _ in range(max_steps):
            if done:
                break
            s, r, done = env.step(int(policy_arr[s]))
            tot += r
        ep_rewards.append(tot)
    return float(np.mean(ep_rewards))


# ── Compounding error demo ────────────────────────────────────────────────────
# Measure return vs trajectory length cap for random-dataset BC vs expert
step_caps    = [1, 2, 5, 10, 20, 50]
bc_random_r  = [evaluate_policy(env, bc_policies['random']['greedy'],
                                 max_steps=cap, seed=7) for cap in step_caps]
bc_expert_r  = [evaluate_policy(env, bc_policies['expert']['greedy'],
                                 max_steps=cap, seed=7) for cap in step_caps]
optimal_r    = [evaluate_policy(env, pi_star, max_steps=cap, seed=7) for cap in step_caps]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(step_caps, bc_random_r,  'o-', color=C_RED,    label='BC (random dataset)',  lw=2)
ax.plot(step_caps, bc_expert_r,  's-', color=C_GREEN,  label='BC (expert dataset)',  lw=2)
ax.plot(step_caps, optimal_r,    '^-', color=C_BLUE,   label='Optimal policy (Q*)',  lw=2)
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.set_xlabel('Max trajectory length', fontsize=11)
ax.set_ylabel('Mean episode return', fontsize=11)
ax.set_title('Figure 2 — BC Performance vs Trajectory Length\n'
             '(Compounding errors degrade random-dataset BC as horizon grows)',
             fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"BC (random) @50 steps : {bc_random_r[-1]:.3f}")
print(f"BC (expert) @50 steps : {bc_expert_r[-1]:.3f}")
print(f"Optimal     @50 steps : {optimal_r[-1]:.3f}")

## 6. Offline Q-Learning and Extrapolation Error

Standard tabular Q-learning applied to the **fixed dataset** (no environment interaction):

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]
$$

The Bellman bootstrap uses $\max_{a'} Q(s', a')$. On a fixed dataset, the maximisation propagates over *all* actions at $s'$, including those with no data support.

In continuous/large discrete spaces this causes **extrapolation error**: Q-values of OOD actions are initialised near zero but get inflated when used as bootstrap targets. The policy then selects these over-estimated OOD actions, collapsing performance.

In the tabular setting we observe this as Q-values diverging for states visited infrequently.

In [ ]:
# ── Offline Q-learning ─────────────────────────────────────────────────────────
def offline_qlearning(dataset, n_states, n_actions, alpha=ALPHA, gamma=GAMMA,
                       n_epochs=20, seed=SEED):
    """Offline (batch) Q-learning: sweep through the dataset repeatedly.

    Returns Q-table (n_states, n_actions) and per-epoch mean |TD error|.
    """
    rng_local = np.random.default_rng(seed)
    Q         = np.zeros((n_states, n_actions))
    td_errors  = []

    data = list(dataset)   # copy so we can shuffle
    for epoch in range(n_epochs):
        rng_local.shuffle(data)
        epoch_errs = []
        for (s, a, r, s2, done) in data:
            target = r + gamma * np.max(Q[s2]) * (1 - done)
            td     = target - Q[s, a]
            Q[s, a] += alpha * td
            epoch_errs.append(abs(td))
        td_errors.append(np.mean(epoch_errs))

    return Q, td_errors


offline_Qs = {}
for name, ds in datasets.items():
    Q_off, errs = offline_qlearning(ds, env.n_states, env.n_actions, n_epochs=30)
    offline_Qs[name] = {'Q': Q_off, 'td_errors': errs}

print("[PASS] Offline Q-learning complete")
for name, res in offline_Qs.items():
    pi_off = np.argmax(res['Q'], axis=1)
    ret    = evaluate_policy(env, pi_off, seed=0)
    print(f"  {name:12s}: mean_return={ret:.3f}  "
          f"final_td_err={res['td_errors'][-1]:.4f}")

In [ ]:
# ── Figure 3: Offline Q-value heatmaps ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ref_vmin = Q_star.max(axis=1).min()
ref_vmax = Q_star.max(axis=1).max()

for ax, (name, res) in zip(axes, offline_Qs.items()):
    Q_off = res['Q']
    V_off = Q_off.max(axis=1).reshape(GRID_SIZE, GRID_SIZE)
    im = ax.imshow(V_off, cmap='RdYlGn', vmin=ref_vmin, vmax=ref_vmax, origin='upper')
    ax.set_title(f'Offline Q-learning\n({name} dataset)', fontsize=10, fontweight='bold')
    for (r, c) in env.obstacles:
        ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                         facecolor=C_RED, alpha=0.7))
    gr, gc = env.goal
    ax.add_patch(mpatches.Rectangle((gc - 0.5, gr - 0.5), 1, 1,
                                     facecolor=C_GREEN, alpha=0.7))
    ax.set_xticks(range(GRID_SIZE))
    ax.set_yticks(range(GRID_SIZE))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Figure 3 — Offline Q-learning: V(s) = max_a Q(s,a) by Dataset Quality\n'
             '(Note distorted values with random dataset due to extrapolation error)',
             fontsize=11)
plt.tight_layout()
plt.show()

## 7. Conservative Q-Learning (CQL)

### 7.1 Theory

**Conservative Q-Learning** (Kumar et al., 2020) augments the standard Bellman loss with a regulariser that *pushes down* Q-values on OOD actions and *pushes up* Q-values on dataset actions:

$$
\boxed{
\mathcal{L}_{\text{CQL}}(Q) = 
\underbrace{\mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\left[\left(r + \gamma \max_{a'} Q(s', a') - Q(s,a)\right)^2\right]}_{\text{standard TD loss}}
+ \alpha \underbrace{\mathbb{E}_{s \sim \mathcal{D}}\left[\log \sum_{a} e^{Q(s,a)} - Q(s, a_{\text{data}})\right]}_{\text{CQL conservative regulariser}}
}
$$

The term $\log \sum_a e^{Q(s,a)} - Q(s, a_{\text{data}})$ is proportional to the gap between the *soft maximum* over all actions and the Q-value of the data action. Minimising it:

- **Reduces** Q-values of all actions (through the log-sum-exp, which acts as a soft upper bound)
- **Increases** Q-values of dataset actions (through the $-Q(s, a_{\text{data}})$ term)

This ensures $Q^{\mu}(s,a) \leq Q^*(s,a)$ for in-distribution actions while keeping OOD actions conservatively valued.

### 7.2 Gradient update (tabular, SGD)

For a single transition $(s, a, r, s')$:

$$
\delta_{\text{TD}} = r + \gamma \max_{a'} Q(s', a') - Q(s,a)
$$
$$
\Delta Q(s,a) = \alpha \left[ \delta_{\text{TD}} - \alpha_{\text{cql}} \left(\text{softmax}_a(Q(s,\cdot))_a - \mathbf{1}[a = a_{\text{data}}]\right) \right]
$$

where $\text{softmax}_a(Q(s,\cdot))_a = e^{Q(s,a)} / \sum_{a''} e^{Q(s,a'')}$ is the softmax weight for action $a$.

In [ ]:
# ── Conservative Q-Learning (CQL) ─────────────────────────────────────────────
def cql_qlearning(dataset, n_states, n_actions, alpha=ALPHA, gamma=GAMMA,
                   alpha_cql=1.0, n_epochs=30, seed=SEED):
    """Tabular CQL with SGD-style updates.

    CQL loss per transition:
        TD loss  + alpha_cql * (log_sum_exp Q(s,·) - Q(s, a_data))

    Gradient w.r.t. Q(s,a):
        -TD_error  + alpha_cql * (softmax(Q(s,·))[a] - 1[a==a_data])

    Returns Q-table, per-epoch TD errors, per-epoch CQL penalties.
    """
    rng_local   = np.random.default_rng(seed)
    Q           = np.zeros((n_states, n_actions))
    td_errors   = []
    cql_penalties = []

    data = list(dataset)
    for epoch in range(n_epochs):
        rng_local.shuffle(data)
        epoch_td  = []
        epoch_cql = []

        for (s, a, r, s2, done) in data:
            # Standard TD error
            target = r + gamma * np.max(Q[s2]) * (1 - done)
            td     = target - Q[s, a]

            # CQL regulariser: softmax over Q(s, ·)
            q_s      = Q[s]
            # numerically stable softmax
            q_s_norm = q_s - q_s.max()
            sm       = np.exp(q_s_norm) / np.exp(q_s_norm).sum()

            # CQL gradient for this action:
            # ∂/∂Q(s,a) [log_sum_exp - Q(s,a_data)]
            #   = softmax[a] - 1[a == a_data]
            cql_grad = sm[a] - 1.0   # since we are only updating Q(s,a)

            # Combined update
            Q[s, a] += alpha * (td - alpha_cql * cql_grad)

            # CQL penalty for logging: log_sum_exp - Q(s, a_data)
            lse     = np.log(np.exp(q_s_norm).sum()) + q_s.max()  # = log_sum_exp(Q(s,·))
            penalty = lse - Q[s, a]
            epoch_td.append(abs(td))
            epoch_cql.append(penalty)

        td_errors.append(np.mean(epoch_td))
        cql_penalties.append(np.mean(epoch_cql))

    return Q, td_errors, cql_penalties


# Run CQL for multiple alpha_cql values on the random dataset (hardest case)
cql_alphas  = [0.0, 0.5, 1.0, 5.0]
cql_results = {}
for ac in cql_alphas:
    Q_cql, td_e, cql_p = cql_qlearning(datasets['random'],
                                         env.n_states, env.n_actions,
                                         alpha_cql=ac, n_epochs=30)
    ret = evaluate_policy(env, np.argmax(Q_cql, axis=1), seed=0)
    cql_results[ac] = {'Q': Q_cql, 'td': td_e, 'cql': cql_p, 'return': ret}

print("[PASS] CQL training complete")
print(f"{'alpha_cql':>12}  {'mean_return':>12}  {'final_td_err':>12}")
for ac, res in cql_results.items():
    print(f"{ac:12.1f}  {res['return']:12.3f}  {res['td'][-1]:12.4f}")

In [ ]:
# ── Figure 4: CQL effect on Q-value distributions ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: V(s) heatmap comparison — offline Q vs CQL(α=1)
ax = axes[0]
V_off = offline_Qs['random']['Q'].max(axis=1).reshape(GRID_SIZE, GRID_SIZE)
V_cql = cql_results[1.0]['Q'].max(axis=1).reshape(GRID_SIZE, GRID_SIZE)
diff  = V_cql - V_off

im = ax.imshow(diff, cmap='RdBu_r', origin='upper',
               vmin=-np.abs(diff).max(), vmax=np.abs(diff).max())
for (r, c) in env.obstacles:
    ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                     facecolor='k', alpha=0.5))
gr, gc = env.goal
ax.add_patch(mpatches.Rectangle((gc - 0.5, gr - 0.5), 1, 1,
                                 facecolor=C_GREEN, alpha=0.5))
ax.set_title('V(s) change: CQL(α=1) − Offline Q\n'
             '(Blue = CQL is more conservative)', fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Right: return vs alpha_cql
ax = axes[1]
alphas  = list(cql_results.keys())
returns = [cql_results[a]['return'] for a in alphas]
bars = ax.bar([str(a) for a in alphas], returns,
              color=[C_RED, C_ORANGE, C_GREEN, C_BLUE])
ax.axhline(evaluate_policy(env, pi_star, seed=0),
           color='k', ls='--', lw=1.5, label='Optimal policy')
ax.set_xlabel('CQL regularisation strength α', fontsize=11)
ax.set_ylabel('Mean episode return', fontsize=11)
ax.set_title('CQL α sweep on random dataset\n'
             '(α=0 is plain offline Q-learning)', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.4)

fig.suptitle('Figure 4 — Conservative Q-Learning', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── CQL vs Offline Q: policy visualisation side by side ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

plot_grid(env, title='Optimal policy (Q*)',
          Q=Q_star, policy=pi_star, ax=axes[0])

pi_off_rand = np.argmax(offline_Qs['random']['Q'], axis=1)
plot_grid(env, title='Offline Q-learning\n(random dataset)',
          Q=offline_Qs['random']['Q'], policy=pi_off_rand, ax=axes[1])

pi_cql = np.argmax(cql_results[1.0]['Q'], axis=1)
plot_grid(env, title='CQL  α=1.0\n(random dataset)',
          Q=cql_results[1.0]['Q'], policy=pi_cql, ax=axes[2])

fig.suptitle('Figure 5 — Policy Comparison: Optimal vs Offline Q vs CQL',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Importance Sampling for Off-Policy Evaluation

**Off-policy evaluation (OPE)** asks: *given a dataset $\mathcal{D}$ generated by $\mu$, estimate the expected return of a different target policy $\pi$.*

### 8.1 Importance Sampling (IS) estimator

For a trajectory $\tau = (s_0, a_0, \ldots, s_T)$:

$$
\rho(\tau) = \prod_{t=0}^{T-1} \frac{\pi(a_t | s_t)}{\mu(a_t | s_t)}
\qquad \text{(importance weight)}
$$

The **IS estimator** is:
$$
\hat{V}_{\text{IS}}(\pi) = \frac{1}{N} \sum_{i=1}^{N} \rho(\tau^{(i)}) \cdot G(\tau^{(i)})
$$

This is **unbiased** but can have **very high variance** when $\pi$ and $\mu$ differ substantially (large $\rho$).

### 8.2 Weighted IS (WIS) estimator

The **self-normalised / weighted IS** estimator reduces variance at the cost of small bias:

$$
\hat{V}_{\text{WIS}}(\pi) = \frac{\sum_{i=1}^{N} \rho(\tau^{(i)}) \cdot G(\tau^{(i)})}{\sum_{i=1}^{N} \rho(\tau^{(i)})}
$$

WIS is consistent and typically has much lower variance than IS.

In [ ]:
# ── Trajectory-based IS/WIS estimators ────────────────────────────────────────
def collect_trajectories(env, policy_fn, n_trajectories=500, max_steps=30,
                          seed=SEED):
    """Collect full trajectories. Returns list of [(s,a,r), ...] per episode."""
    rng_local = np.random.default_rng(seed)
    trajs = []
    for _ in range(n_trajectories):
        s    = env.reset()
        done = False
        traj = []
        for _ in range(max_steps):
            if done:
                break
            a            = policy_fn(s, rng_local)
            s2, r, done  = env.step(a)
            traj.append((s, a, r))
            s = s2
        trajs.append(traj)
    return trajs


def is_estimate(trajs, pi_target, pi_behavior, gamma=GAMMA, clip=10.0):
    """IS and WIS off-policy value estimates.

    Args:
        trajs       -- list of [(s,a,r), ...]
        pi_target   -- (n_states, n_actions) target policy probabilities
        pi_behavior -- (n_states, n_actions) behavior policy probabilities
        clip        -- clip individual IS weights to reduce extreme values

    Returns:
        is_val  -- IS estimate
        wis_val -- WIS estimate
        weights -- per-trajectory IS weights
        returns -- per-trajectory discounted returns
    """
    weights = []
    returns = []

    for traj in trajs:
        rho = 1.0
        G   = 0.0
        for t, (s, a, r) in enumerate(traj):
            p_pi  = pi_target[s, a]
            p_mu  = pi_behavior[s, a]
            p_mu  = max(p_mu, 1e-8)   # avoid division by zero
            rho  *= p_pi / p_mu
            G    += (gamma ** t) * r

        rho = min(rho, clip)   # clip to reduce variance
        weights.append(rho)
        returns.append(G)

    weights = np.array(weights)
    returns = np.array(returns)

    is_val  = float(np.mean(weights * returns))
    wis_val = float(np.sum(weights * returns) / (np.sum(weights) + 1e-10))
    return is_val, wis_val, weights, returns


# Build stochastic policy representations
# -- behavior: epsilon-optimal (ε=0.3)
pi_beh_stoch = bc_policies['eps_optimal']['pi']   # fitted from eps_optimal data

# -- target: CQL policy (make stochastic via one-hot)
pi_cql_stoch = np.zeros((env.n_states, env.n_actions))
for s_idx in range(env.n_states):
    pi_cql_stoch[s_idx, pi_cql[s_idx]] = 1.0

# -- target: optimal policy
pi_star_stoch = np.zeros((env.n_states, env.n_actions))
for s_idx in range(env.n_states):
    pi_star_stoch[s_idx, pi_star[s_idx]] = 1.0

# Collect trajectories from behavior policy for evaluation
beh_trajs = collect_trajectories(env, eps_optimal_policy, n_trajectories=1000,
                                  seed=SEED+10)

# IS and WIS estimates
is_cql, wis_cql, w_cql, r_cql       = is_estimate(beh_trajs, pi_cql_stoch, pi_beh_stoch)
is_opt, wis_opt, w_opt, r_opt       = is_estimate(beh_trajs, pi_star_stoch, pi_beh_stoch)
is_beh, wis_beh, w_beh, r_beh       = is_estimate(beh_trajs, pi_beh_stoch,  pi_beh_stoch)

# Ground-truth returns by direct rollout
true_cql  = evaluate_policy(env, pi_cql,   seed=5)
true_opt  = evaluate_policy(env, pi_star,  seed=5)
true_beh  = evaluate_policy(env, np.argmax(pi_beh_stoch, axis=1), seed=5)

print("[PASS] IS/WIS estimates complete")
print(f"{'Policy':>10}  {'IS est':>8}  {'WIS est':>8}  {'True ret':>9}")
print(f"{'behavior':>10}  {is_beh:8.3f}  {wis_beh:8.3f}  {true_beh:9.3f}")
print(f"{'CQL':>10}  {is_cql:8.3f}  {wis_cql:8.3f}  {true_cql:9.3f}")
print(f"{'optimal':>10}  {is_opt:8.3f}  {wis_opt:8.3f}  {true_opt:9.3f}")

In [ ]:
# ── IS variance analysis across sample sizes ──────────────────────────────────
def is_variance_experiment(env, pi_target, pi_behavior_fn, pi_behavior_mat,
                            sample_sizes, n_repeats=30, seed=SEED):
    """Compute IS and WIS variance as a function of number of trajectories."""
    rng_local = np.random.default_rng(seed)
    is_vars, wis_vars = [], []

    for n in sample_sizes:
        is_vals, wis_vals = [], []
        for rep in range(n_repeats):
            trajs = collect_trajectories(env, pi_behavior_fn, n_trajectories=n,
                                          seed=int(rng_local.integers(1e6)))
            iv, wv, _, _ = is_estimate(trajs, pi_target, pi_behavior_mat)
            is_vals.append(iv)
            wis_vals.append(wv)
        is_vars.append(np.var(is_vals))
        wis_vars.append(np.var(wis_vals))

    return np.array(is_vars), np.array(wis_vars)


sample_sizes = [20, 50, 100, 200, 500]
is_v, wis_v  = is_variance_experiment(env, pi_star_stoch, eps_optimal_policy,
                                        pi_beh_stoch, sample_sizes, n_repeats=40)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: variance vs sample size
axes[0].loglog(sample_sizes, is_v,  'o-', color=C_RED,   lw=2, label='IS variance')
axes[0].loglog(sample_sizes, wis_v, 's-', color=C_BLUE,  lw=2, label='WIS variance')
axes[0].set_xlabel('Number of trajectories (log scale)', fontsize=10)
axes[0].set_ylabel('Estimator variance (log scale)', fontsize=10)
axes[0].set_title('IS vs WIS Variance\n(estimating optimal policy value)', fontsize=10)
axes[0].legend(fontsize=10)
axes[0].grid(True, which='both', alpha=0.3)

# Right: IS weight distribution
axes[1].hist(w_opt[w_opt < np.percentile(w_opt, 95)], bins=30,
             color=C_PURPLE, alpha=0.7, edgecolor='white', label='IS weights')
axes[1].axvline(w_opt.mean(), color=C_RED, lw=2, ls='--',
                label=f'mean={w_opt.mean():.2f}')
axes[1].set_xlabel('Importance weight ρ(τ)', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)
axes[1].set_title('IS Weight Distribution\n(95th-percentile clipped)', fontsize=10)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Figure 6 — Importance Sampling: Variance Analysis',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"[PASS] IS variance experiment: IS_var@500={is_v[-1]:.4f}, WIS_var@500={wis_v[-1]:.4f}")

## 9. Distributional Shift Analysis

The **coverage** of the dataset determines which $(s,a)$ pairs can be reliably learned from. When the target policy $\pi$ visits states or takes actions not covered by the behavior policy $\mu$, the learned Q-function must extrapolate — and extrapolation errors compound.

We visualise:
1. Where each dataset covers the state space (visitation frequency)
2. How dataset quality impacts the final policy performance
3. The gap between behavior policy distribution and optimal policy distribution

In [ ]:
# ── Compute state visitation frequencies ──────────────────────────────────────
def state_visitation(dataset, n_states):
    """Compute normalised state visitation frequency from a dataset."""
    counts = np.zeros(n_states)
    for (s, a, r, s2, done) in dataset:
        counts[s] += 1
    total = counts.sum()
    return counts / (total + 1e-10)


def optimal_visitation(env, policy_arr, n_rollouts=2000, max_steps=30):
    """Empirical state visitation of a deterministic policy via rollouts."""
    counts = np.zeros(env.n_states)
    for ep in range(n_rollouts):
        s = env.reset()
        for _ in range(max_steps):
            counts[s] += 1
            s, r, done = env.step(int(policy_arr[s]))
            if done:
                break
    return counts / (counts.sum() + 1e-10)


sv_random      = state_visitation(datasets['random'],      env.n_states)
sv_eps_optimal = state_visitation(datasets['eps_optimal'], env.n_states)
sv_expert      = state_visitation(datasets['expert'],      env.n_states)
sv_optimal_pi  = optimal_visitation(env, pi_star)

print("[PASS] State visitation computed")

In [ ]:
# ── Figure 7: Distributional shift visualisation ───────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

vis_data = [
    (sv_random,      'Dataset coverage\n(random behavior)',       axes[0, 0]),
    (sv_eps_optimal, 'Dataset coverage\n(ε-optimal behavior)',    axes[0, 1]),
    (sv_expert,      'Dataset coverage\n(expert behavior)',       axes[0, 2]),
    (sv_optimal_pi,  'Optimal policy\nstate visitation',          axes[1, 0]),
    (sv_expert - sv_random,
     'Coverage gap:\nexpert − random\n(warm=better expert coverage)',
     axes[1, 1]),
    (sv_optimal_pi - sv_random,
     'Distributional shift:\noptimal policy − random dataset\n(warm=OOD for random)',
     axes[1, 2]),
]

for sv, title, ax in vis_data:
    grid_sv = sv.reshape(GRID_SIZE, GRID_SIZE)
    abs_max  = max(np.abs(grid_sv).max(), 1e-6)
    if sv.min() < -1e-6:   # difference plot
        im = ax.imshow(grid_sv, cmap='RdBu_r', origin='upper',
                       vmin=-abs_max, vmax=abs_max)
    else:
        im = ax.imshow(grid_sv, cmap='YlOrRd', origin='upper', vmin=0)

    # Overlay obstacle and goal markers
    for (r, c) in env.obstacles:
        ax.text(c, r, 'X', ha='center', va='center', fontsize=13,
                fontweight='bold', color='white')
    gr, gc = env.goal
    ax.text(gc, gr, 'G', ha='center', va='center', fontsize=13,
            fontweight='bold', color='white')

    ax.set_title(title, fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(GRID_SIZE))
    ax.set_yticks(range(GRID_SIZE))

fig.suptitle('Figure 7 — Distributional Shift Analysis\n'
             'Top row: state visitation in offline datasets | '
             'Bottom row: coverage vs policy need',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Dataset quality experiment ─────────────────────────────────────────────────
# Vary dataset size and measure CQL policy performance
sizes    = [500, 1000, 2000, 4000, 8000]
methods  = ['offline_q', 'cql_05', 'cql_10', 'bc']
results  = {m: [] for m in methods}

for sz in sizes:
    ds_small = datasets['random'][:sz]

    # Offline Q
    Q_o, _ = offline_qlearning(ds_small, env.n_states, env.n_actions, n_epochs=20)
    results['offline_q'].append(evaluate_policy(env, np.argmax(Q_o, axis=1), seed=1))

    # CQL α=0.5
    Q_c5, _, _ = cql_qlearning(ds_small, env.n_states, env.n_actions,
                                 alpha_cql=0.5, n_epochs=20)
    results['cql_05'].append(evaluate_policy(env, np.argmax(Q_c5, axis=1), seed=1))

    # CQL α=1.0
    Q_c1, _, _ = cql_qlearning(ds_small, env.n_states, env.n_actions,
                                 alpha_cql=1.0, n_epochs=20)
    results['cql_10'].append(evaluate_policy(env, np.argmax(Q_c1, axis=1), seed=1))

    # BC
    pi_bc_s, _ = behavior_clone(ds_small, env.n_states, env.n_actions)
    results['bc'].append(evaluate_policy(env, greedy_policy_from_pi(pi_bc_s), seed=1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(sizes, results['offline_q'], 'o-', color=C_RED,    lw=2,
            label='Offline Q-learning')
ax.semilogx(sizes, results['cql_05'],   's-', color=C_ORANGE,  lw=2,
            label='CQL  (α=0.5)')
ax.semilogx(sizes, results['cql_10'],   '^-', color=C_GREEN,   lw=2,
            label='CQL  (α=1.0)')
ax.semilogx(sizes, results['bc'],       'D-', color=C_PURPLE,  lw=2,
            label='Behavior Cloning')
ax.axhline(evaluate_policy(env, pi_star, seed=1), color='k', ls='--', lw=1.5,
           label='Optimal policy')
ax.set_xlabel('Dataset size (log scale)', fontsize=11)
ax.set_ylabel('Mean episode return', fontsize=11)
ax.set_title('Figure 8 — Policy Performance vs Dataset Size\n'
             '(random behavior dataset)', fontsize=11)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print("[PASS] Dataset quality experiment complete")

## 10. Summary

### Results table

| Method | Dataset | Mean return | Notes |
|--------|---------|-------------|-------|
| Online Q-learning | — (live) | *(reference)* | Requires environment access |
| Behavior Cloning | random | low | Compounding errors; no RL objective |
| Behavior Cloning | expert | near-optimal | Works well only with expert data |
| Offline Q-learning | random | low/unstable | Extrapolation error dominates |
| Offline Q-learning | expert | good | Sufficient coverage reduces error |
| CQL (α=1.0) | random | improved | Conservative regularisation helps |
| CQL (α=1.0) | expert | best offline | Combines coverage + conservatism |

### Key takeaways

1. **Distributional shift is the fundamental challenge** in offline RL. The behavior policy determines which (s,a) pairs are reliably estimated, and any method that bootstraps on OOD actions will suffer.

2. **Behavior Cloning is a strong baseline** with expert data but degrades with poor datasets and compounds errors over long horizons.

3. **Offline Q-learning extrapolation error** is severe with random/poor datasets. The Bellman max operator propagates inflated OOD estimates throughout the value function.

4. **CQL directly addresses extrapolation** by adding a conservative regulariser that lower-bounds in-distribution Q-values and suppresses OOD Q-values. The hyperparameter α trades off conservatism vs. performance.

5. **Importance sampling** enables unbiased off-policy evaluation but suffers from high variance when the target and behavior policies diverge. WIS reduces variance at the cost of slight bias.

6. **Dataset quality matters enormously**: expert or near-expert data yields much better offline policies across all methods.

## References

1. Kumar, A., Zhou, A., Tucker, G., & Levine, S. (2020). **Conservative Q-Learning for Offline Reinforcement Learning.** *NeurIPS 2020.* [arXiv:2006.04779](https://arxiv.org/abs/2006.04779)

2. Levine, S., Kumar, A., Tucker, G., & Fu, J. (2020). **Offline Reinforcement Learning: Tutorial, Review, and Perspectives on Open Problems.** [arXiv:2005.01643](https://arxiv.org/abs/2005.01643)

3. Precup, D., Sutton, R. S., & Singh, S. (2000). **Eligibility Traces for Off-Policy Policy Evaluation.** *ICML 2000.*

4. Lange, S., Gabel, T., & Riedmiller, M. (2012). **Batch Reinforcement Learning.** In *Reinforcement Learning: State of the Art*, Springer.

5. Sutton, R. S., & Barto, A. G. (2018). **Reinforcement Learning: An Introduction** (2nd ed.). MIT Press. Chapters 5–6 (off-policy methods).

6. Fujimoto, S., Meger, D., & Precup, D. (2019). **Off-Policy Deep Reinforcement Learning without Exploration.** *ICML 2019.* [arXiv:1812.02900](https://arxiv.org/abs/1812.02900) (BCQ)

7. Thomas, P., & Brunskill, E. (2016). **Data-Efficient Off-Policy Policy Evaluation for Reinforcement Learning.** *ICML 2016.*